In [0]:
# %python
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# 1. Speed data
speed_df = pd.DataFrame([
    {"model": "(LG) OLED G4", "CP": -0.82855, "LGC": -0.33792, "System": -0.72699, "Overall": -0.68980},
    {"model": "(LG) OLED C4", "CP": -0.92659, "LGC": -0.39433, "System": -0.58208, "Overall": -0.68233},
    {"model": "(LG) OLED B3", "CP":  0.029287, "LGC": -1.18993, "System": -0.21975, "Overall": -0.31417},
    {"model": "(LG) OLED B4", "CP": -0.52754, "LGC":  0.175649, "System": -0.35050, "Overall": -0.31609},
    {"model": "(LG) OLED C3", "CP": -0.35434, "LGC": -0.85923, "System": -0.35050, "Overall": -0.45378},
    {"model": "(LG) UR80",    "CP":  1.268824, "LGC":  1.331085, "System":  1.710002, "Overall":  1.457748},
    {"model": "(LG) QNED80",  "CP":  1.338917, "LGC":  1.274678, "System":  0.519824, "Overall":  0.998432},
    {"model": "(SS) S95C",    "CP":  0.547737, "LGC": -0.97611, "System":  0.005639, "Overall":  0.026129},
])

# 2. VOC data
voc_df = pd.DataFrame([
    {"model": "(LG) OLED B3", "VOC Group": "TV Overall", "avg_score": 0.49, "review_cnt": 35,  "pos_cnt": 26,  "neg_cnt": 9},
    {"model": "(LG) OLED B4", "VOC Group": "TV Overall", "avg_score": 0.22, "review_cnt": 55,  "pos_cnt": 33,  "neg_cnt": 21},
    {"model": "(LG) OLED C3", "VOC Group": "TV Overall", "avg_score": 0.93, "review_cnt": 138, "pos_cnt": 133, "neg_cnt": 5},
    {"model": "(LG) OLED C4", "VOC Group": "TV Overall", "avg_score": 0.74, "review_cnt": 99,  "pos_cnt": 86,  "neg_cnt": 13},
    {"model": "(LG) OLED G4", "VOC Group": "TV Overall", "avg_score": 0.93, "review_cnt": 57,  "pos_cnt": 55,  "neg_cnt": 2},
    {"model": "(LG) QNED80",  "VOC Group": "TV Overall", "avg_score": 0.32, "review_cnt": 233, "pos_cnt": 153, "neg_cnt": 79},
    {"model": "(LG) UR80",    "VOC Group": "TV Overall", "avg_score": 0.16, "review_cnt": 311, "pos_cnt": 179, "neg_cnt": 128},
    {"model": "(SS) S95C",    "VOC Group": "TV Overall", "avg_score": 0.43, "review_cnt": 23,  "pos_cnt": 16,  "neg_cnt": 6},

    {"model": "(LG) OLED B3", "VOC Group": "App/Web Overall", "avg_score": 0.24, "review_cnt": 21,  "pos_cnt": 13, "neg_cnt": 8},
    {"model": "(LG) OLED B4", "VOC Group": "App/Web Overall", "avg_score": -0.04,"review_cnt": 28,  "pos_cnt": 13, "neg_cnt": 14},
    {"model": "(LG) OLED C3", "VOC Group": "App/Web Overall", "avg_score": 0.51, "review_cnt": 111, "pos_cnt": 84, "neg_cnt": 27},
    {"model": "(LG) OLED C4", "VOC Group": "App/Web Overall", "avg_score": 0.38, "review_cnt": 63,  "pos_cnt": 43, "neg_cnt": 19},
    {"model": "(LG) OLED G4", "VOC Group": "App/Web Overall", "avg_score": 0.60, "review_cnt": 15,  "pos_cnt": 11, "neg_cnt": 2},
    {"model": "(LG) QNED80",  "VOC Group": "App/Web Overall", "avg_score": 0.45, "review_cnt": 138, "pos_cnt": 99, "neg_cnt": 37},
    {"model": "(LG) UR80",    "VOC Group": "App/Web Overall", "avg_score": -0.18,"review_cnt": 161, "pos_cnt": 65, "neg_cnt": 94},
    {"model": "(SS) S95C",    "VOC Group": "App/Web Overall", "avg_score": 0.41, "review_cnt": 27,  "pos_cnt": 19, "neg_cnt": 8},
])

# 3. Pivot VOC and merge
voc_pivot = voc_df.pivot(index="model", columns="VOC Group", values="avg_score").reset_index()
df = speed_df.merge(voc_pivot, on="model", how="left")

speed_cols = ["CP", "LGC", "System", "Overall"]
voc_cols = ["TV Overall", "App/Web Overall"]

# 4. Residual and gap
coef_tv = np.polyfit(df["Overall"], df["TV Overall"], 1)
coef_app = np.polyfit(df["Overall"], df["App/Web Overall"], 1)

df["TV Predicted"] = coef_tv[0] * df["Overall"] + coef_tv[1]
df["App/Web Predicted"] = coef_app[0] * df["Overall"] + coef_app[1]
df["TV Residual"] = df["TV Overall"] - df["TV Predicted"]
df["App/Web Residual"] = df["App/Web Overall"] - df["App/Web Predicted"]
df["VOC Gap"] = df["TV Overall"] - df["App/Web Overall"]

# 5. Correlation table
corr_df = df[speed_cols + voc_cols].corr().loc[speed_cols, voc_cols]
print("Correlation between Speed Metrics and VOC")
display(corr_df.round(3))

print("Model Summary")
display(df[["model", "CP", "LGC", "System", "Overall", "TV Overall", "App/Web Overall", "TV Residual", "App/Web Residual", "VOC Gap"]].round(3))

# 6. Visualization 1: 4x2 subplot
sns.set_theme(style="whitegrid", font="DejaVu Sans")
fig, axes = plt.subplots(len(speed_cols), len(voc_cols), figsize=(14, 18))

for i, speed in enumerate(speed_cols):
    for j, voc in enumerate(voc_cols):
        ax = axes[i, j]
        color = "steelblue" if voc == "TV Overall" else "darkorange"
        sns.regplot(
            data=df,
            x=speed,
            y=voc,
            ax=ax,
            scatter_kws={"s": 90},
            line_kws={"color": "tomato"},
            color=color
        )
        for _, r in df.iterrows():
            ax.text(r[speed] + 0.02, r[voc] + 0.01, r["model"], fontsize=8)
        corr = df[speed].corr(df[voc])
        ax.set_title(f"{speed} vs {voc}\nCorrelation: {corr:.2f}")
        ax.set_xlabel(f"{speed} Score")
        ax.set_ylabel(f"{voc} Score")

plt.tight_layout()
plt.show()

# 7. Visualization 2: heatmap + key scatter + residual + gap
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sns.heatmap(corr_df, annot=True, cmap="RdBu_r", center=0, ax=axes[0, 0])
axes[0, 0].set_title("Correlation Heatmap: Speed vs VOC")

sns.regplot(data=df, x="Overall", y="TV Overall", ax=axes[0, 1], scatter_kws={"s": 100}, line_kws={"color": "tomato"})
for _, r in df.iterrows():
    axes[0, 1].text(r["Overall"] + 0.02, r["TV Overall"] + 0.01, r["model"], fontsize=8)
axes[0, 1].set_title("Overall Speed vs TV Overall VOC")

residual_df = df.melt(
    id_vars="model",
    value_vars=["TV Residual", "App/Web Residual"],
    var_name="Residual Type",
    value_name="Residual"
)
sns.barplot(data=residual_df, x="Residual", y="model", hue="Residual Type", ax=axes[1, 0])
axes[1, 0].axvline(0, color="black", linestyle="--", linewidth=1)
axes[1, 0].set_title("Residuals by Model")

gap_df = df.sort_values("VOC Gap", ascending=False)
sns.barplot(data=gap_df, x="VOC Gap", y="model", color="slateblue", ax=axes[1, 1])
axes[1, 1].axvline(0, color="black", linestyle="--", linewidth=1)
axes[1, 1].set_title("TV Overall - App/Web Overall VOC Gap")

plt.tight_layout()
plt.show()

# 8. Visualization 3: review volume
review_pivot = voc_df.pivot(index="model", columns="VOC Group", values="review_cnt").reset_index()
review_long = review_pivot.melt(id_vars="model", var_name="VOC Group", value_name="Review Count")

plt.figure(figsize=(10, 6))
sns.barplot(data=review_long, x="Review Count", y="model", hue="VOC Group")
plt.title("Review Count by Model and VOC Group")
plt.tight_layout()
plt.show()


In [0]:
# %python
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# 1. Speed data
speed_df = pd.DataFrame([
    {"model": "(LG) OLED G4", "CP": -0.82855, "LGC": -0.33792, "System": -0.72699, "Overall": -0.68980},
    {"model": "(LG) OLED C4", "CP": -0.92659, "LGC": -0.39433, "System": -0.58208, "Overall": -0.68233},
    {"model": "(LG) OLED B3", "CP":  0.029287, "LGC": -1.18993, "System": -0.21975, "Overall": -0.31417},
    {"model": "(LG) OLED B4", "CP": -0.52754, "LGC":  0.175649, "System": -0.35050, "Overall": -0.31609},
    {"model": "(LG) OLED C3", "CP": -0.35434, "LGC": -0.85923, "System": -0.35050, "Overall": -0.45378},
    {"model": "(LG) UR80",    "CP":  1.268824, "LGC":  1.331085, "System":  1.710002, "Overall":  1.457748},
    {"model": "(LG) QNED80",  "CP":  1.338917, "LGC":  1.274678, "System":  0.519824, "Overall":  0.998432},
    {"model": "(SS) S95C",    "CP":  0.547737, "LGC": -0.97611, "System":  0.005639, "Overall":  0.026129},
])

# 2. VOC data
voc_df = pd.DataFrame([
    {"model": "(LG) OLED B3", "VOC Group": "TV Overall", "avg_score": 0.49},
    {"model": "(LG) OLED B4", "VOC Group": "TV Overall", "avg_score": 0.22},
    {"model": "(LG) OLED C3", "VOC Group": "TV Overall", "avg_score": 0.93},
    {"model": "(LG) OLED C4", "VOC Group": "TV Overall", "avg_score": 0.74},
    {"model": "(LG) OLED G4", "VOC Group": "TV Overall", "avg_score": 0.93},
    {"model": "(LG) QNED80",  "VOC Group": "TV Overall", "avg_score": 0.32},
    {"model": "(LG) UR80",    "VOC Group": "TV Overall", "avg_score": 0.16},
    {"model": "(SS) S95C",    "VOC Group": "TV Overall", "avg_score": 0.43},

    {"model": "(LG) OLED B3", "VOC Group": "App/Web Overall", "avg_score": 0.24},
    {"model": "(LG) OLED B4", "VOC Group": "App/Web Overall", "avg_score": -0.04},
    {"model": "(LG) OLED C3", "VOC Group": "App/Web Overall", "avg_score": 0.51},
    {"model": "(LG) OLED C4", "VOC Group": "App/Web Overall", "avg_score": 0.38},
    {"model": "(LG) OLED G4", "VOC Group": "App/Web Overall", "avg_score": 0.60},
    {"model": "(LG) QNED80",  "VOC Group": "App/Web Overall", "avg_score": 0.45},
    {"model": "(LG) UR80",    "VOC Group": "App/Web Overall", "avg_score": -0.18},
    {"model": "(SS) S95C",    "VOC Group": "App/Web Overall", "avg_score": 0.41},
])

# 3. Merge
voc_pivot = voc_df.pivot(index="model", columns="VOC Group", values="avg_score").reset_index()
df = speed_df.merge(voc_pivot, on="model", how="left")

# 4. Exclude UR80
df = df[df["model"] != "(LG) UR80"].copy()

speed_cols = ["CP", "LGC", "System", "Overall"]
voc_cols = ["TV Overall", "App/Web Overall"]

# 5. Residual and gap
coef_tv = np.polyfit(df["Overall"], df["TV Overall"], 1)
coef_app = np.polyfit(df["Overall"], df["App/Web Overall"], 1)

df["TV Predicted"] = coef_tv[0] * df["Overall"] + coef_tv[1]
df["App/Web Predicted"] = coef_app[0] * df["Overall"] + coef_app[1]
df["TV Residual"] = df["TV Overall"] - df["TV Predicted"]
df["App/Web Residual"] = df["App/Web Overall"] - df["App/Web Predicted"]
df["VOC Gap"] = df["TV Overall"] - df["App/Web Overall"]

# 6. Correlation
corr_df = df[speed_cols + voc_cols].corr().loc[speed_cols, voc_cols]
print("Correlation excluding (LG) UR80")
display(corr_df.round(3))

print("Model Summary excluding (LG) UR80")
display(df[["model", "CP", "LGC", "System", "Overall", "TV Overall", "App/Web Overall", "TV Residual", "App/Web Residual", "VOC Gap"]].round(3))

# 7. 4x2 subplot
sns.set_theme(style="whitegrid", font="DejaVu Sans")
fig, axes = plt.subplots(len(speed_cols), len(voc_cols), figsize=(14, 18))

for i, speed in enumerate(speed_cols):
    for j, voc in enumerate(voc_cols):
        ax = axes[i, j]
        color = "steelblue" if voc == "TV Overall" else "darkorange"
        sns.regplot(
            data=df,
            x=speed,
            y=voc,
            ax=ax,
            scatter_kws={"s": 90},
            line_kws={"color": "tomato"},
            color=color
        )
        for _, r in df.iterrows():
            ax.text(r[speed] + 0.02, r[voc] + 0.01, r["model"], fontsize=8)
        corr = df[speed].corr(df[voc])
        ax.set_title(f"{speed} vs {voc}\nCorrelation: {corr:.2f}")
        ax.set_xlabel(f"{speed} Score")
        ax.set_ylabel(f"{voc} Score")

plt.tight_layout()
plt.show()

# 8. Heatmap + key charts
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sns.heatmap(corr_df, annot=True, cmap="RdBu_r", center=0, ax=axes[0, 0])
axes[0, 0].set_title("Correlation Heatmap Excluding UR80")

sns.regplot(data=df, x="Overall", y="TV Overall", ax=axes[0, 1], scatter_kws={"s": 100}, line_kws={"color": "tomato"})
for _, r in df.iterrows():
    axes[0, 1].text(r["Overall"] + 0.02, r["TV Overall"] + 0.01, r["model"], fontsize=8)
axes[0, 1].set_title("Overall Speed vs TV Overall VOC Excluding UR80")

residual_df = df.melt(
    id_vars="model",
    value_vars=["TV Residual", "App/Web Residual"],
    var_name="Residual Type",
    value_name="Residual"
)
sns.barplot(data=residual_df, x="Residual", y="model", hue="Residual Type", ax=axes[1, 0])
axes[1, 0].axvline(0, color="black", linestyle="--", linewidth=1)
axes[1, 0].set_title("Residuals by Model Excluding UR80")

gap_df = df.sort_values("VOC Gap", ascending=False)
sns.barplot(data=gap_df, x="VOC Gap", y="model", color="slateblue", ax=axes[1, 1])
axes[1, 1].axvline(0, color="black", linestyle="--", linewidth=1)
axes[1, 1].set_title("TV Overall - App/Web Overall VOC Gap Excluding UR80")

plt.tight_layout()
plt.show()


In [0]:
# %python
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.DataFrame([
    {"model": "(LG) OLED G4", "CP": -0.82855419, "LGC": -0.33792,  "System": -0.72699, "Overall": -0.6898,   "TV Overall": 0.93, "App/Web Overall": 0.60},
    {"model": "(LG) OLED C4", "CP": -0.92659416, "LGC": -0.39433,  "System": -0.58208, "Overall": -0.68233,  "TV Overall": 0.74, "App/Web Overall": 0.38},
    {"model": "(LG) OLED B3", "CP":  0.029286973,"LGC": -1.18993,  "System": -0.21975, "Overall": -0.31417,  "TV Overall": 0.49, "App/Web Overall": 0.24},
    {"model": "(LG) OLED B4", "CP": -0.52753699, "LGC":  0.175649, "System": -0.3505,  "Overall": -0.31609,  "TV Overall": 0.22, "App/Web Overall": -0.04},
    {"model": "(LG) OLED C3", "CP": -0.35434235, "LGC": -0.85923,  "System": -0.3505,  "Overall": -0.45378,  "TV Overall": 0.93, "App/Web Overall": 0.51},
    {"model": "(LG) UR80",    "CP":  1.268823807,"LGC":  1.331085, "System":  1.710002,"Overall":  1.457748, "TV Overall": 0.16, "App/Web Overall": -0.18},
    {"model": "(LG) QNED80",  "CP":  1.338916913,"LGC":  1.274678, "System":  0.519824,"Overall":  0.998432, "TV Overall": 0.32, "App/Web Overall": 0.45},
])

speed_cols = ["CP", "LGC", "System", "Overall"]
voc_cols = ["TV Overall", "App/Web Overall"]

sns.set_theme(style="whitegrid", font="DejaVu Sans")
fig, axes = plt.subplots(len(speed_cols), len(voc_cols), figsize=(14, 18))

for i, speed in enumerate(speed_cols):
    for j, voc in enumerate(voc_cols):
        ax = axes[i, j]
        sns.regplot(
            data=df,
            x=speed,
            y=voc,
            ax=ax,
            scatter_kws={"s": 90},
            line_kws={"color": "tomato"}
        )

        for _, r in df.iterrows():
            ax.text(r[speed] + 0.02, r[voc] + 0.01, r["model"], fontsize=8)

        corr = df[speed].corr(df[voc])
        ax.set_title(f"{speed} vs {voc}\nCorrelation: {corr:.2f}")
        ax.set_xlabel(f"{speed} Score")
        ax.set_ylabel(f"{voc} Score")

plt.tight_layout()
plt.show()


In [0]:
# %python
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# 1. Speed data
speed_df = pd.DataFrame([
    {"model": "(LG) OLED G4", "CP": -0.82855, "LGC": -0.33792, "System": -0.72699, "Overall": -0.68980},
    {"model": "(LG) OLED C4", "CP": -0.92659, "LGC": -0.39433, "System": -0.58208, "Overall": -0.68233},
    {"model": "(LG) OLED B3", "CP":  0.029287, "LGC": -1.18993, "System": -0.21975, "Overall": -0.31417},
    {"model": "(LG) OLED B4", "CP": -0.52754, "LGC":  0.175649, "System": -0.35050, "Overall": -0.31609},
    {"model": "(LG) OLED C3", "CP": -0.35434, "LGC": -0.85923, "System": -0.35050, "Overall": -0.45378},
    {"model": "(LG) UR80",    "CP":  1.268824, "LGC":  1.331085, "System":  1.710002, "Overall":  1.457748},
    {"model": "(LG) QNED80",  "CP":  1.338917, "LGC":  1.274678, "System":  0.519824, "Overall":  0.998432},
    {"model": "(SS) S95C",    "CP":  0.547737, "LGC": -0.97611, "System":  0.005639, "Overall":  0.026129},
])

# 2. VOC data
voc_df = pd.DataFrame([
    {"model": "(LG) OLED B3", "VOC Group": "TV Overall", "avg_score": 0.49},
    {"model": "(LG) OLED B4", "VOC Group": "TV Overall", "avg_score": 0.22},
    {"model": "(LG) OLED C3", "VOC Group": "TV Overall", "avg_score": 0.93},
    {"model": "(LG) OLED C4", "VOC Group": "TV Overall", "avg_score": 0.74},
    {"model": "(LG) OLED G4", "VOC Group": "TV Overall", "avg_score": 0.93},
    {"model": "(LG) QNED80",  "VOC Group": "TV Overall", "avg_score": 0.32},
    {"model": "(LG) UR80",    "VOC Group": "TV Overall", "avg_score": 0.16},
    {"model": "(SS) S95C",    "VOC Group": "TV Overall", "avg_score": 0.43},

    {"model": "(LG) OLED B3", "VOC Group": "App/Web Overall", "avg_score": 0.24},
    {"model": "(LG) OLED B4", "VOC Group": "App/Web Overall", "avg_score": -0.04},
    {"model": "(LG) OLED C3", "VOC Group": "App/Web Overall", "avg_score": 0.51},
    {"model": "(LG) OLED C4", "VOC Group": "App/Web Overall", "avg_score": 0.38},
    {"model": "(LG) OLED G4", "VOC Group": "App/Web Overall", "avg_score": 0.60},
    {"model": "(LG) QNED80",  "VOC Group": "App/Web Overall", "avg_score": 0.45},
    {"model": "(LG) UR80",    "VOC Group": "App/Web Overall", "avg_score": -0.18},
    {"model": "(SS) S95C",    "VOC Group": "App/Web Overall", "avg_score": 0.41},
])

# 3. Merge
voc_pivot = voc_df.pivot(index="model", columns="VOC Group", values="avg_score").reset_index()
df = speed_df.merge(voc_pivot, on="model", how="left")

# 4. Exclude UR80
df = df[df["model"] != "(LG) UR80"].copy()

speed_cols = ["CP", "LGC", "System", "Overall"]
voc_cols = ["TV Overall", "App/Web Overall"]

# 5. Residual and gap
coef_tv = np.polyfit(df["Overall"], df["TV Overall"], 1)
coef_app = np.polyfit(df["Overall"], df["App/Web Overall"], 1)

df["TV Predicted"] = coef_tv[0] * df["Overall"] + coef_tv[1]
df["App/Web Predicted"] = coef_app[0] * df["Overall"] + coef_app[1]
df["TV Residual"] = df["TV Overall"] - df["TV Predicted"]
df["App/Web Residual"] = df["App/Web Overall"] - df["App/Web Predicted"]
df["VOC Gap"] = df["TV Overall"] - df["App/Web Overall"]

# 6. Correlation
corr_df = df[speed_cols + voc_cols].corr().loc[speed_cols, voc_cols]
print("Correlation excluding (LG) UR80")
display(corr_df.round(3))

print("Model Summary excluding (LG) UR80")
display(df[["model", "CP", "LGC", "System", "Overall", "TV Overall", "App/Web Overall", "TV Residual", "App/Web Residual", "VOC Gap"]].round(3))

# 7. 4x2 subplot
sns.set_theme(style="whitegrid", font="DejaVu Sans")
fig, axes = plt.subplots(len(speed_cols), len(voc_cols), figsize=(14, 18))

for i, speed in enumerate(speed_cols):
    for j, voc in enumerate(voc_cols):
        ax = axes[i, j]
        color = "steelblue" if voc == "TV Overall" else "darkorange"
        sns.regplot(
            data=df,
            x=speed,
            y=voc,
            ax=ax,
            scatter_kws={"s": 90},
            line_kws={"color": "tomato"},
            color=color
        )
        for _, r in df.iterrows():
            ax.text(r[speed] + 0.02, r[voc] + 0.01, r["model"], fontsize=8)
        corr = df[speed].corr(df[voc])
        ax.set_title(f"{speed} vs {voc}\nCorrelation: {corr:.2f}")
        ax.set_xlabel(f"{speed} Score")
        ax.set_ylabel(f"{voc} Score")

plt.tight_layout()
plt.show()

# 8. Heatmap + key charts
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sns.heatmap(corr_df, annot=True, cmap="RdBu_r", center=0, ax=axes[0, 0])
axes[0, 0].set_title("Correlation Heatmap Excluding UR80")

sns.regplot(data=df, x="Overall", y="TV Overall", ax=axes[0, 1], scatter_kws={"s": 100}, line_kws={"color": "tomato"})
for _, r in df.iterrows():
    axes[0, 1].text(r["Overall"] + 0.02, r["TV Overall"] + 0.01, r["model"], fontsize=8)
axes[0, 1].set_title("Overall Speed vs TV Overall VOC Excluding UR80")

residual_df = df.melt(
    id_vars="model",
    value_vars=["TV Residual", "App/Web Residual"],
    var_name="Residual Type",
    value_name="Residual"
)
sns.barplot(data=residual_df, x="Residual", y="model", hue="Residual Type", ax=axes[1, 0])
axes[1, 0].axvline(0, color="black", linestyle="--", linewidth=1)
axes[1, 0].set_title("Residuals by Model Excluding UR80")

gap_df = df.sort_values("VOC Gap", ascending=False)
sns.barplot(data=gap_df, x="VOC Gap", y="model", color="slateblue", ax=axes[1, 1])
axes[1, 1].axvline(0, color="black", linestyle="--", linewidth=1)
axes[1, 1].set_title("TV Overall - App/Web Overall VOC Gap Excluding UR80")

plt.tight_layout()
plt.show()


In [0]:
# %python
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# 1. Source data
speed_df = pd.DataFrame([
    {"model": "(LG) OLED G4", "Overall": -0.68980},
    {"model": "(LG) OLED C4", "Overall": -0.68233},
    {"model": "(LG) OLED B3", "Overall": -0.31417},
    {"model": "(LG) OLED B4", "Overall": -0.31609},
    {"model": "(LG) OLED C3", "Overall": -0.45378},
    {"model": "(LG) UR80",    "Overall":  1.457748},
    {"model": "(LG) QNED80",  "Overall":  0.998432},
    {"model": "(SS) S95C",    "Overall":  0.026129},
])

voc_df = pd.DataFrame([
    {"model": "(LG) OLED G4", "TV Overall": 0.93, "App/Web Overall": 0.60},
    {"model": "(LG) OLED C4", "TV Overall": 0.74, "App/Web Overall": 0.38},
    {"model": "(LG) OLED B3", "TV Overall": 0.49, "App/Web Overall": 0.24},
    {"model": "(LG) OLED B4", "TV Overall": 0.22, "App/Web Overall": -0.04},
    {"model": "(LG) OLED C3", "TV Overall": 0.93, "App/Web Overall": 0.51},
    {"model": "(LG) UR80",    "TV Overall": 0.16, "App/Web Overall": -0.18},
    {"model": "(LG) QNED80",  "TV Overall": 0.32, "App/Web Overall": 0.45},
    {"model": "(SS) S95C",    "TV Overall": 0.43, "App/Web Overall": 0.41},
])

df = speed_df.merge(voc_df, on="model", how="left")

def make_residual_df(input_df, label):
    work = input_df.copy()

    coef_tv = np.polyfit(work["Overall"], work["TV Overall"], 1)
    coef_app = np.polyfit(work["Overall"], work["App/Web Overall"], 1)

    work["TV Predicted"] = coef_tv[0] * work["Overall"] + coef_tv[1]
    work["App Predicted"] = coef_app[0] * work["Overall"] + coef_app[1]

    work["TV Residual"] = work["TV Overall"] - work["TV Predicted"]
    work["App/Web Residual"] = work["App/Web Overall"] - work["App Predicted"]
    work["Case"] = label

    return work[["model", "Case", "TV Residual", "App/Web Residual"]]

resid_incl = make_residual_df(df, "Including UR80")
resid_excl = make_residual_df(df[df["model"] != "(LG) UR80"], "Excluding UR80")

resid_all = pd.concat([resid_incl, resid_excl], ignore_index=True)
plot_df = resid_all.melt(
    id_vars=["model", "Case"],
    value_vars=["TV Residual", "App/Web Residual"],
    var_name="Residual Type",
    value_name="Residual"
)

# 2. Plot
sns.set_theme(style="whitegrid", font="DejaVu Sans")
fig, axes = plt.subplots(1, 2, figsize=(16, 8), sharex=True)

for ax, case in zip(axes, ["Including UR80", "Excluding UR80"]):
    temp = plot_df[plot_df["Case"] == case].copy()
    sns.barplot(
        data=temp,
        x="Residual",
        y="model",
        hue="Residual Type",
        ax=ax
    )
    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.set_title(f"Residual by Model: {case}")
    ax.set_xlabel("Actual VOC - Predicted VOC")
    ax.set_ylabel("Model")

plt.tight_layout()
plt.show()


In [0]:
# %python
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Source data
df = pd.DataFrame([
    {"model": "(LG) OLED G4", "TV Overall": 0.93, "App/Web Overall": 0.60},
    {"model": "(LG) OLED C4", "TV Overall": 0.74, "App/Web Overall": 0.38},
    {"model": "(LG) OLED B3", "TV Overall": 0.49, "App/Web Overall": 0.24},
    {"model": "(LG) OLED B4", "TV Overall": 0.22, "App/Web Overall": -0.04},
    {"model": "(LG) OLED C3", "TV Overall": 0.93, "App/Web Overall": 0.51},
    {"model": "(LG) UR80",    "TV Overall": 0.16, "App/Web Overall": -0.18},
    {"model": "(LG) QNED80",  "TV Overall": 0.32, "App/Web Overall": 0.45},
    {"model": "(SS) S95C",    "TV Overall": 0.43, "App/Web Overall": 0.41},
])

# 2. Calculate gap
df["VOC Gap"] = df["TV Overall"] - df["App/Web Overall"]

# 3. Sort for plotting
plot_df = df.sort_values("VOC Gap", ascending=False).copy()

# 4. Plot
sns.set_theme(style="whitegrid", font="DejaVu Sans")
plt.figure(figsize=(10, 6))

ax = sns.barplot(
    data=plot_df,
    x="VOC Gap",
    y="model",
    color="slateblue"
)

ax.axvline(0, color="black", linestyle="--", linewidth=1)
ax.set_title("VOC Gap by Model")
ax.set_xlabel("TV Overall VOC - App/Web Overall VOC")
ax.set_ylabel("Model")

for i, row in plot_df.reset_index(drop=True).iterrows():
    ax.text(row["VOC Gap"] + 0.01, i, f'{row["VOC Gap"]:.2f}', va="center")

plt.tight_layout()
plt.show()


In [0]:
# %python
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# 1. Model-level analysis table
df = pd.DataFrame([
    {
        "platform": "webos24", "DEVICE_NAME": "o24", "model": "(LG) OLED G4",
        "HOME_raw_second": 0.7, "HOME_speed_norm": -0.71608, "Overall_speed": -0.68980,
        "CP_speed": -0.82855, "LGC_speed": -0.33792, "System_speed": -0.72699,
        "avg_home_entry_cnt": 54.02, "avg_home_active_days": 14.08,
        "avg_home_duration_sec": 17.62, "avg_bounce_5s_rate": 0.4379, "avg_bounce_10s_rate": 0.6913
    },
    {
        "platform": "webos24", "DEVICE_NAME": "o22n2", "model": "(LG) OLED C4",
        "HOME_raw_second": 0.7, "HOME_speed_norm": -0.71608, "Overall_speed": -0.68233,
        "CP_speed": -0.92659, "LGC_speed": -0.39433, "System_speed": -0.58208,
        "avg_home_entry_cnt": 48.69, "avg_home_active_days": 13.15,
        "avg_home_duration_sec": 18.27, "avg_bounce_5s_rate": 0.4182, "avg_bounce_10s_rate": 0.6742
    },
    {
        "platform": "webos23", "DEVICE_NAME": "o22n", "model": "(LG) OLED C3",
        "HOME_raw_second": 0.7, "HOME_speed_norm": -0.33050, "Overall_speed": -0.45378,
        "CP_speed": -0.35434, "LGC_speed": -0.85923, "System_speed": -0.35050,
        "avg_home_entry_cnt": 53.95, "avg_home_active_days": 14.02,
        "avg_home_duration_sec": 18.36, "avg_bounce_5s_rate": 0.4332, "avg_bounce_10s_rate": 0.6718
    },
    {
        "platform": "webos24", "DEVICE_NAME": "k24", "model": "(LG) OLED B4",
        "HOME_raw_second": 0.9, "HOME_speed_norm": -0.33050, "Overall_speed": -0.31609,
        "CP_speed": -0.52754, "LGC_speed": 0.175649, "System_speed": -0.35050,
        "avg_home_entry_cnt": 43.86, "avg_home_active_days": 12.51,
        "avg_home_duration_sec": 19.64, "avg_bounce_5s_rate": 0.3301, "avg_bounce_10s_rate": 0.6043
    },
    {
        "platform": "webos23", "DEVICE_NAME": "k8hpp", "model": "(LG) OLED B3",
        "HOME_raw_second": 0.9, "HOME_speed_norm": -0.71608, "Overall_speed": -0.31417,
        "CP_speed": 0.029287, "LGC_speed": -1.18993, "System_speed": -0.21975,
        "avg_home_entry_cnt": 51.44, "avg_home_active_days": 14.21,
        "avg_home_duration_sec": 19.26, "avg_bounce_5s_rate": 0.3653, "avg_bounce_10s_rate": 0.6167
    },
    {
        "platform": "webos24", "DEVICE_NAME": "k8lpn2", "model": "(LG) QNED80",
        "HOME_raw_second": 2.0, "HOME_speed_norm": 1.019035, "Overall_speed": 0.998432,
        "CP_speed": 1.338917, "LGC_speed": 1.274678, "System_speed": 0.519824,
        "avg_home_entry_cnt": 40.91, "avg_home_active_days": 12.84,
        "avg_home_duration_sec": 23.95, "avg_bounce_5s_rate": 0.2386, "avg_bounce_10s_rate": 0.4692
    },
    {
        "platform": "webos23", "DEVICE_NAME": "k8lpn", "model": "(LG) UR80",
        "HOME_raw_second": 1.6, "HOME_speed_norm": 1.790197, "Overall_speed": 1.457748,
        "CP_speed": 1.268824, "LGC_speed": 1.331085, "System_speed": 1.710002,
        "avg_home_entry_cnt": 38.39, "avg_home_active_days": 11.00,
        "avg_home_duration_sec": 24.21, "avg_bounce_5s_rate": 0.2033, "avg_bounce_10s_rate": 0.4513
    },
])

display(df)

# 2. Correlation summary
x_cols = ["HOME_raw_second", "HOME_speed_norm", "Overall_speed"]
y_cols = ["avg_home_duration_sec", "avg_home_entry_cnt", "avg_bounce_5s_rate", "avg_bounce_10s_rate", "avg_home_active_days"]

corr_rows = []
for x in x_cols:
    row = {"X": x}
    for y in y_cols:
        row[y] = df[x].corr(df[y])
    corr_rows.append(row)

corr_df = pd.DataFrame(corr_rows)
print("Correlation Summary")
display(corr_df.round(3))

# 3. 3 x 5 subplot
sns.set_theme(style="whitegrid", font="DejaVu Sans")
fig, axes = plt.subplots(len(x_cols), len(y_cols), figsize=(22, 14))

for i, x in enumerate(x_cols):
    for j, y in enumerate(y_cols):
        ax = axes[i, j]
        sns.regplot(
            data=df,
            x=x,
            y=y,
            ax=ax,
            scatter_kws={"s": 100},
            line_kws={"color": "tomato"}
        )

        for _, r in df.iterrows():
            ax.text(r[x] + 0.01, r[y], r["DEVICE_NAME"], fontsize=8)

        corr = df[x].corr(df[y])
        ax.set_title(f"{x} vs {y}\nCorrelation: {corr:.2f}")
        ax.set_xlabel(x)
        ax.set_ylabel(y)

plt.tight_layout()
plt.show()

# 4. Main chart set for reporting
main_pairs = [
    ("HOME_raw_second", "avg_bounce_5s_rate"),
    ("HOME_raw_second", "avg_bounce_10s_rate"),
    ("HOME_raw_second", "avg_home_duration_sec"),
    ("HOME_raw_second", "avg_home_active_days"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (x, y) in zip(axes.flatten(), main_pairs):
    sns.regplot(
        data=df,
        x=x,
        y=y,
        ax=ax,
        scatter_kws={"s": 120},
        line_kws={"color": "tomato"}
    )
    for _, r in df.iterrows():
        ax.text(r[x] + 0.02, r[y], r["DEVICE_NAME"], fontsize=8)
    corr = df[x].corr(df[y])
    ax.set_title(f"{x} vs {y}\nCorrelation: {corr:.2f}")

plt.tight_layout()
plt.show()

# 5. Including / Excluding UR80 comparison
df_excl = df[df["DEVICE_NAME"] != "k8lpn"].copy()

compare_rows = []
for label, temp in [("Including UR80", df), ("Excluding UR80", df_excl)]:
    for y in y_cols:
        compare_rows.append({
            "Case": label,
            "Metric": y,
            "Corr_with_HOME_raw": temp["HOME_raw_second"].corr(temp[y]),
            "Corr_with_HOME_norm": temp["HOME_speed_norm"].corr(temp[y]),
            "Corr_with_Overall_speed": temp["Overall_speed"].corr(temp[y]),
        })

compare_df = pd.DataFrame(compare_rows)
print("Including / Excluding UR80 Comparison")
display(compare_df.round(3))

# 6. Focus comparison chart
focus_metrics = ["avg_bounce_5s_rate", "avg_bounce_10s_rate", "avg_home_duration_sec", "avg_home_active_days"]

fig, axes = plt.subplots(2, 4, figsize=(22, 10))

for col_idx, metric in enumerate(focus_metrics):
    # including
    ax1 = axes[0, col_idx]
    sns.regplot(
        data=df,
        x="HOME_raw_second",
        y=metric,
        ax=ax1,
        scatter_kws={"s": 120},
        line_kws={"color": "tomato"}
    )
    for _, r in df.iterrows():
        ax1.text(r["HOME_raw_second"] + 0.02, r[metric], r["DEVICE_NAME"], fontsize=8)
    ax1.set_title(f"Including UR80\nHOME_raw_second vs {metric}\nCorr: {df['HOME_raw_second'].corr(df[metric]):.2f}")

    # excluding
    ax2 = axes[1, col_idx]
    sns.regplot(
        data=df_excl,
        x="HOME_raw_second",
        y=metric,
        ax=ax2,
        scatter_kws={"s": 120},
        line_kws={"color": "tomato"}
    )
    for _, r in df_excl.iterrows():
        ax2.text(r["HOME_raw_second"] + 0.02, r[metric], r["DEVICE_NAME"], fontsize=8)
    ax2.set_title(f"Excluding UR80\nHOME_raw_second vs {metric}\nCorr: {df_excl['HOME_raw_second'].corr(df_excl[metric]):.2f}")

plt.tight_layout()
plt.show()
